# Round 4 Counterparty Signal Miner

This notebook exhaustively searches buyer → seller counterparty relationships, including grouped counterparties, for `HYDROGEL_PACK`, `VELVETFRUIT_EXTRACT`, and the `VEV_*` voucher contracts.

The core question is:

> When a named buyer trades with a named seller, does the product tend to move up or down afterward?

The notebook scores both:
- **LONG_AFTER_TRADE**: buy after the observed trade, exit after a fixed horizon.
- **SHORT_AFTER_TRADE**: short after the observed trade, exit after a fixed horizon.

It also includes plotting utilities to manually review any relationship on top of the price path.


In [ ]:
import pandas as pd
import numpy as np
import itertools
import re
import glob
from pathlib import Path
import matplotlib.pyplot as plt

DATA_DIR = Path("/mnt/data")

PRICE_FILES = sorted(DATA_DIR.glob("prices_round_4_day_*.csv"))
TRADE_FILES = sorted(DATA_DIR.glob("trades_round_4_day_*.csv"))

# Forward horizons in timestamp units. Competition data usually ticks every 100.
HORIZONS = [100, 200, 500, 1000, 2000, 5000, 10000]

# Keep all non-empty groups. With 7 Marks this is only 127 buyer groups x 127 seller groups.
# Set MAX_GROUP_SIZE = 1 for only individual pairs, 2 for pairs/groups of size <=2, etc.
MAX_GROUP_SIZE = None

# Minimum filters for final candidate table.
MIN_EVENTS = 10
MIN_DAYS = 2

PRICE_FILES, TRADE_FILES


## 1. Load and sanity-check the data

In [ ]:
def day_from_path(path):
    return int(re.search(r"day_(\d+)", str(path)).group(1))

prices = []
for f in PRICE_FILES:
    df = pd.read_csv(f, sep=";")
    prices.append(df)
prices = pd.concat(prices, ignore_index=True)

trades = []
for f in TRADE_FILES:
    df = pd.read_csv(f, sep=";")
    df["day"] = day_from_path(f)
    trades.append(df)
trades = pd.concat(trades, ignore_index=True).rename(columns={"symbol": "product"})

marks = sorted(set(trades["buyer"].dropna()).union(trades["seller"].dropna()))

print("prices:", prices.shape)
print("trades:", trades.shape)
print("products:", sorted(prices["product"].unique()))
print("marks:", marks)
print()
display(trades.groupby(["product"]).size().sort_values().to_frame("trade_count"))
display(trades.groupby(["buyer", "seller", "product"]).size().sort_values(ascending=False).head(30).to_frame("n"))


## 2. Attach forward returns to every trade

For each trade, this creates:

\[
\text{forward move}_{H} = \text{mid}(t+H) - \text{mid}(t)
\]

Then the simple event score is:

\[
\text{long points}_{H} = \text{quantity} \times (\text{mid}(t+H)-\text{mid}(t))
\]

This is not yet a true executable PnL because it ignores spread, fill, position limits, and overlapping signals. It is the right first scan for whether a counterparty event predicts direction.


In [ ]:
mid = prices[["day", "timestamp", "product", "mid_price"]].copy()

events = trades.merge(
    mid.rename(columns={"mid_price": "mid_t"}),
    on=["day", "timestamp", "product"],
    how="left",
)

for H in HORIZONS:
    future_mid = mid.copy()
    future_mid["timestamp"] -= H
    events = events.merge(
        future_mid.rename(columns={"mid_price": f"mid_fwd_{H}"}),
        on=["day", "timestamp", "product"],
        how="left",
    )
    events[f"ret_{H}"] = events[f"mid_fwd_{H}"] - events["mid_t"]
    events[f"qty_ret_{H}"] = events["quantity"] * events[f"ret_{H}"]

display(events.head())
display(events[[f"ret_{H}" for H in HORIZONS]].describe())


## 3. Exhaustive grouped buyer → seller scan

This is the important part.

With 7 counterparties, every non-empty group is feasible:

- 7 individual groups
- 21 two-Mark groups
- 35 three-Mark groups
- …
- 127 total non-empty groups

So the notebook checks every:

\[
\text{buyer group} \rightarrow \text{seller group}
\]

for every product and horizon.

Implementation trick: for each product/day/horizon, create a 7×7 buyer-seller matrix, then compute all group sums by matrix multiplication:

\[
G \cdot M \cdot G^T
\]

where `G` is the group-membership matrix.


In [ ]:
def build_group_masks(marks, max_group_size=None):
    n = len(marks)
    masks = []
    for mask in range(1, 1 << n):
        size = int(mask.bit_count())
        if max_group_size is None or size <= max_group_size:
            masks.append(mask)
    masks = np.array(masks, dtype=int)
    G = np.array([[(mask >> i) & 1 for i in range(n)] for mask in masks], dtype=float)
    mask_to_names = {
        int(mask): tuple(marks[i] for i in range(n) if (int(mask) >> i) & 1)
        for mask in masks
    }
    return masks, G, mask_to_names

masks, G, mask_to_names = build_group_masks(marks, MAX_GROUP_SIZE)
mark_idx = {m: i for i, m in enumerate(marks)}

def mask_label(mask):
    return "|".join(mask_to_names[int(mask)])

print("Number of groups:", len(masks))
print("Number of buyer_group -> seller_group combinations:", len(masks) ** 2)


In [ ]:
def score_all_group_relationships(events, products=None, horizons=HORIZONS):
    if products is None:
        products = sorted(events["product"].unique())

    rows = []
    n_marks = len(marks)

    for product in products:
        eprod = events[events["product"] == product]

        for H in horizons:
            col = f"ret_{H}"

            for day in sorted(eprod["day"].unique()):
                sub = eprod[(eprod["day"] == day) & eprod[col].notna()]

                sum_m = np.zeros((n_marks, n_marks), dtype=float)
                cnt_m = np.zeros((n_marks, n_marks), dtype=float)
                qty_m = np.zeros((n_marks, n_marks), dtype=float)

                for r in sub.itertuples(index=False):
                    bi = mark_idx[r.buyer]
                    si = mark_idx[r.seller]
                    q = float(r.quantity)
                    val = float(getattr(r, col)) * q

                    sum_m[bi, si] += val
                    cnt_m[bi, si] += 1
                    qty_m[bi, si] += q

                # All grouped buyer/seller combinations at once.
                sum_g = G @ sum_m @ G.T
                cnt_g = G @ cnt_m @ G.T
                qty_g = G @ qty_m @ G.T

                nonzero = np.argwhere(cnt_g > 0)
                for gi, gj in nonzero:
                    rows.append(
                        (
                            product,
                            H,
                            day,
                            int(masks[gi]),
                            int(masks[gj]),
                            int(cnt_g[gi, gj]),
                            float(qty_g[gi, gj]),
                            float(sum_g[gi, gj]),
                        )
                    )

    out = pd.DataFrame(
        rows,
        columns=[
            "product",
            "horizon",
            "day",
            "buyer_mask",
            "seller_mask",
            "n_events",
            "qty",
            "long_pnl_points",
        ],
    )
    return out

scores_day = score_all_group_relationships(events)

display(scores_day.head())
print("day-level grouped rows:", len(scores_day))


## 4. Aggregate and rank candidate signals

A relationship is ranked separately for long and short direction.

Important columns:
- `score`: quantity-weighted total forward points.
- `avg_per_event`: score divided by number of events.
- `worst_day`: worst day-level score for that direction.
- `robust_score`: total score with a penalty/reward for worst day.
- `days`: number of days where this relationship appeared.

A signal with a lower total score but positive score on all three days is usually more trustworthy than a huge one-day overfit.


In [ ]:
def aggregate_scores(scores_day):
    agg = (
        scores_day.groupby(["product", "horizon", "buyer_mask", "seller_mask"])
        .agg(
            n_events=("n_events", "sum"),
            days=("day", "nunique"),
            qty=("qty", "sum"),
            long_pnl_points=("long_pnl_points", "sum"),
            day_pnl_mean=("long_pnl_points", "mean"),
            day_pnl_min=("long_pnl_points", "min"),
            day_pnl_max=("long_pnl_points", "max"),
        )
        .reset_index()
    )

    long = agg.copy()
    long["direction"] = "LONG_AFTER_TRADE"
    long["score"] = long["long_pnl_points"]
    long["avg_per_event"] = long["score"] / long["n_events"]
    long["worst_day"] = long["day_pnl_min"]

    short = agg.copy()
    short["direction"] = "SHORT_AFTER_TRADE"
    short["score"] = -short["long_pnl_points"]
    short["avg_per_event"] = short["score"] / short["n_events"]
    short["worst_day"] = -short["day_pnl_max"]

    out = pd.concat([long, short], ignore_index=True)
    out["buyer_group"] = out["buyer_mask"].map(mask_label)
    out["seller_group"] = out["seller_mask"].map(mask_label)
    out["buyer_group_size"] = out["buyer_mask"].map(lambda x: int(int(x).bit_count()))
    out["seller_group_size"] = out["seller_mask"].map(lambda x: int(int(x).bit_count()))
    out["robust_score"] = out["score"] + 0.5 * out["worst_day"]
    return out

all_signals = aggregate_scores(scores_day)

candidates = all_signals[
    (all_signals["n_events"] >= MIN_EVENTS)
    & (all_signals["days"] >= MIN_DAYS)
    & (all_signals["score"] > 0)
].copy()

rank_cols = [
    "product",
    "horizon",
    "direction",
    "buyer_group",
    "seller_group",
    "n_events",
    "days",
    "score",
    "avg_per_event",
    "worst_day",
    "robust_score",
]

top_candidates = candidates.sort_values(
    ["days", "robust_score", "score"],
    ascending=[False, False, False],
).reset_index(drop=True)

display(top_candidates[rank_cols].head(50))

# Save compact output files.
top_candidates[rank_cols + ["buyer_mask", "seller_mask", "buyer_group_size", "seller_group_size"]].to_csv(
    DATA_DIR / "round4_top_group_counterparty_signals.csv",
    index=False,
)

individual_candidates = all_signals[
    (all_signals["buyer_group_size"] == 1)
    & (all_signals["seller_group_size"] == 1)
    & (all_signals["n_events"] >= 3)
    & (all_signals["score"] > 0)
].sort_values(["days", "robust_score", "score"], ascending=[False, False, False])

individual_candidates[rank_cols].to_csv(
    DATA_DIR / "round4_individual_pair_counterparty_signals.csv",
    index=False,
)

print("Saved:")
print(DATA_DIR / "round4_top_group_counterparty_signals.csv")
print(DATA_DIR / "round4_individual_pair_counterparty_signals.csv")


## 5. Individual pair view

This strips out grouped signals and shows the cleanest Mark A → Mark B relationships.

This is where you want to look first, because grouped signals often just rediscover an individual pair plus irrelevant extra Marks.


In [ ]:
display(individual_candidates[rank_cols].head(60))


## 6. De-duplicate grouped signals into minimal relationships

Grouped scans often produce many equivalent supersets. Example:

`{Mark 55, Mark 67} -> {Mark 14}`

may score the same as:

`{Mark 55, Mark 67, Mark 01} -> {Mark 14}`

if `Mark 01 -> Mark 14` never occurs for that product.

This cell keeps the strongest candidates but drops group supersets with the exact same event count and score as a smaller contained group.


In [ ]:
def is_subset_mask(a, b):
    # Is group a contained in group b?
    return (int(a) & int(b)) == int(a)

def minimal_group_candidates(df, max_rows=5000):
    work = df.sort_values(
        ["product", "horizon", "direction", "score", "n_events"],
        ascending=[True, True, True, False, False],
    ).copy()

    kept = []
    for row in work.head(max_rows).itertuples(index=False):
        dominated = False
        for k in kept:
            same_context = (
                row.product == k.product
                and row.horizon == k.horizon
                and row.direction == k.direction
            )
            same_stats = (
                row.n_events == k.n_events
                and abs(row.score - k.score) < 1e-9
            )
            contained = (
                is_subset_mask(k.buyer_mask, row.buyer_mask)
                and is_subset_mask(k.seller_mask, row.seller_mask)
            )
            if same_context and same_stats and contained:
                dominated = True
                break
        if not dominated:
            kept.append(row)

    return pd.DataFrame(kept)

minimal_candidates = minimal_group_candidates(top_candidates, max_rows=10000)
display(minimal_candidates[rank_cols].head(100))


## 7. Plot any relationship for manual review

The plot shows the mid-price path and marks every trade that matches the relationship.

Interpretation:
- For `LONG_AFTER_TRADE`, upward moves after markers are good.
- For `SHORT_AFTER_TRADE`, downward moves after markers are good.
- Check whether the signal fires before the move, not after the move is already obvious.
- Check if it only works on one day.


In [ ]:
def parse_group(group):
    if isinstance(group, str):
        return tuple(x.strip() for x in group.split("|") if x.strip())
    return tuple(group)

def matching_events(product, buyer_group, seller_group):
    buyer_group = set(parse_group(buyer_group))
    seller_group = set(parse_group(seller_group))
    return events[
        (events["product"] == product)
        & (events["buyer"].isin(buyer_group))
        & (events["seller"].isin(seller_group))
    ].copy()

def plot_signal(product, buyer_group, seller_group, direction="LONG_AFTER_TRADE", horizon=1000, day=None):
    sig = matching_events(product, buyer_group, seller_group)
    if day is not None:
        sig = sig[sig["day"] == day]

    days = sorted(sig["day"].unique())
    if not days:
        print("No matching events.")
        return

    for d in days:
        p = prices[(prices["product"] == product) & (prices["day"] == d)].sort_values("timestamp")
        s = sig[sig["day"] == d].sort_values("timestamp")

        plt.figure(figsize=(14, 5))
        plt.plot(p["timestamp"], p["mid_price"], linewidth=1)

        plt.scatter(s["timestamp"], s["mid_t"], marker="o", s=50)

        for _, r in s.iterrows():
            x0 = r["timestamp"]
            x1 = r["timestamp"] + horizon
            y0 = r["mid_t"]
            y1 = r.get(f"mid_fwd_{horizon}", np.nan)
            if pd.notna(y1):
                plt.plot([x0, x1], [y0, y1], linewidth=1, alpha=0.4)

        plt.title(
            f"{product} day {d}: {direction}\n"
            f"buyer_group={buyer_group}  seller_group={seller_group}  horizon={horizon}"
        )
        plt.xlabel("timestamp")
        plt.ylabel("mid_price")
        plt.grid(True, alpha=0.25)
        plt.show()

        # cumulative event score for this day
        s = s[s[f"ret_{horizon}"].notna()].copy()
        signed = 1 if direction == "LONG_AFTER_TRADE" else -1
        s["event_points"] = signed * s["quantity"] * s[f"ret_{horizon}"]
        s["cum_points"] = s["event_points"].cumsum()

        plt.figure(figsize=(14, 4))
        plt.plot(s["timestamp"], s["cum_points"], marker="o")
        plt.title(f"Cumulative event score: {product} day {d}, horizon={horizon}")
        plt.xlabel("timestamp")
        plt.ylabel("quantity-weighted points")
        plt.grid(True, alpha=0.25)
        plt.show()

# Example: plot the top minimal candidate.
r = minimal_candidates.iloc[0]
plot_signal(
    product=r["product"],
    buyer_group=r["buyer_group"],
    seller_group=r["seller_group"],
    direction=r["direction"],
    horizon=int(r["horizon"]),
)


## 8. Relationship heatmaps

These heatmaps show individual buyer → seller scores for one product/horizon/direction.

Use this to see whether a group signal is really several Marks behaving similarly, or just one pair dominating.


In [ ]:
def relationship_matrix(product, horizon=1000, direction="LONG_AFTER_TRADE", day=None):
    sub = events[(events["product"] == product) & events[f"ret_{horizon}"].notna()]
    if day is not None:
        sub = sub[sub["day"] == day]

    M = pd.DataFrame(0.0, index=marks, columns=marks)
    C = pd.DataFrame(0, index=marks, columns=marks)

    signed = 1 if direction == "LONG_AFTER_TRADE" else -1

    for r in sub.itertuples(index=False):
        val = signed * float(r.quantity) * float(getattr(r, f"ret_{horizon}"))
        M.loc[r.buyer, r.seller] += val
        C.loc[r.buyer, r.seller] += 1

    return M, C

def plot_relationship_heatmap(product, horizon=1000, direction="LONG_AFTER_TRADE", day=None, min_count=1):
    M, C = relationship_matrix(product, horizon, direction, day)
    A = M.copy()
    A[C < min_count] = np.nan

    plt.figure(figsize=(8, 6))
    plt.imshow(A.values, aspect="auto")
    plt.xticks(range(len(marks)), marks, rotation=45, ha="right")
    plt.yticks(range(len(marks)), marks)
    plt.colorbar(label="quantity-weighted forward points")
    plt.title(f"{product}: {direction}, horizon={horizon}, day={day}, min_count={min_count}")
    plt.xlabel("seller")
    plt.ylabel("buyer")
    plt.tight_layout()
    plt.show()

    display(M)
    display(C)

plot_relationship_heatmap("VELVETFRUIT_EXTRACT", horizon=1000, direction="LONG_AFTER_TRADE", min_count=3)


## 9. Cross-product signal scan

Sometimes a trade in one product predicts another product.

Examples worth checking:
- `VELVETFRUIT_EXTRACT` trades predicting `VEV_*` option moves.
- `VEV_*` trades predicting `VELVETFRUIT_EXTRACT` moves.
- voucher trades at one strike predicting nearby strikes.

This scan keeps the same buyer/seller relationship but lets the target product differ from the traded product.


In [ ]:
def attach_cross_product_returns(trades, prices, target_products, horizons=HORIZONS):
    base = trades.copy()
    out = base.copy()

    for target in target_products:
        target_mid = prices[prices["product"] == target][["day", "timestamp", "mid_price"]].copy()
        out = out.merge(
            target_mid.rename(columns={"mid_price": f"{target}__mid_t"}),
            on=["day", "timestamp"],
            how="left",
        )
        for H in horizons:
            fut = target_mid.copy()
            fut["timestamp"] -= H
            out = out.merge(
                fut.rename(columns={"mid_price": f"{target}__mid_fwd_{H}"}),
                on=["day", "timestamp"],
                how="left",
            )
            out[f"{target}__ret_{H}"] = out[f"{target}__mid_fwd_{H}"] - out[f"{target}__mid_t"]

    return out

# Cross-product scan for individual pairs first.
target_products = sorted(prices["product"].unique())
cross = attach_cross_product_returns(trades, prices, target_products)

cross_rows = []
for source_product in sorted(trades["product"].unique()):
    src = cross[cross["product"] == source_product]
    for target in target_products:
        for H in HORIZONS:
            col = f"{target}__ret_{H}"
            sub = src[src[col].notna()]
            for (buyer, seller), g in sub.groupby(["buyer", "seller"]):
                long_score = (g["quantity"] * g[col]).sum()
                n = len(g)
                days = g["day"].nunique()
                cross_rows.append((source_product, target, H, buyer, seller, n, days, "LONG_AFTER_TRADE", long_score, long_score / n))
                cross_rows.append((source_product, target, H, buyer, seller, n, days, "SHORT_AFTER_TRADE", -long_score, -long_score / n))

cross_scores = pd.DataFrame(
    cross_rows,
    columns=["source_product", "target_product", "horizon", "buyer", "seller", "n_events", "days", "direction", "score", "avg_per_event"],
)

cross_candidates = cross_scores[
    (cross_scores["score"] > 0)
    & (cross_scores["n_events"] >= 5)
    & (cross_scores["days"] >= 2)
    & (cross_scores["source_product"] != cross_scores["target_product"])
].sort_values(["days", "score"], ascending=[False, False])

display(cross_candidates.head(100))
cross_candidates.to_csv(DATA_DIR / "round4_cross_product_counterparty_signals.csv", index=False)
print(DATA_DIR / "round4_cross_product_counterparty_signals.csv")


## 10. Validation checklist before using a signal in `Trader.run`

Do not directly trade the top table without these checks:

1. **Day split:** the signal should work on all three days, or at least not catastrophically fail on one.
2. **Causality:** the marker should appear before the move, not because the trade happened at the moved price.
3. **Spread:** replace mid-to-mid with executable ask-to-bid or bid-to-ask PnL.
4. **Overlap:** if the same signal fires repeatedly, cap position and avoid double-counting the same move.
5. **Position limits:** simulate inventory with the actual product limits.
6. **Latency assumption:** in Prosperity, you see market trades from the previous state, so test a one-tick delay before entering.
7. **Interaction with current strategy:** counterparty alpha should bias fair price or add small directional orders, not fight your existing market making unless the edge is huge.
8. **Overfit groups:** prefer minimal groups. If a group works only because it includes one individual pair, implement the individual pair.
